In [1]:
# Define parameters
id = 'stars'
project_dirname = 'cheatgrass-spread'
species_name = 'Cheatgrass'
species_lookup = 'bromus tectorum'
sample_filename = 'cheatgrass-stars-data'
gbif_filename = 'gbif_cheatgrass.csv'
plot_filename = 'cheatgrass-spread'
plot_height = 500

In [2]:
# Import libraries
import os
import pathlib
import shutil
import time
import zipfile
from getpass import getpass
from glob import glob

import earthpy
import pandas as pd
import pygbif.occurrences as occ
import pygbif.species as species

In [3]:
# Create data directory
project = earthpy.Project(dirname='cheatgrass-spread')
# Download sample data
project.get_data()

# Display the project directory
project.project_dir

PosixPath('/workspaces/data/cheatgrass-spread')

In [4]:
####--------------------------####
#### DO NOT MODIFY THIS CODE! ####
####--------------------------####
# This code ASKS for your credentials and saves it for the rest of the session.
# NEVER put your credentials into your code!!!!

# GBIF needs a username, password, and email -- all need to match the account
reset = False

# Request and store username
if (not ('GBIF_USER'  in os.environ)) or reset:
    os.environ['GBIF_USER'] = input('GBIF username:')

# Securely request and store password
if (not ('GBIF_PWD'  in os.environ)) or reset:
    os.environ['GBIF_PWD'] = getpass('GBIF password:')
    
# Request and store account email address
if (not ('GBIF_EMAIL'  in os.environ)) or reset:
    os.environ['GBIF_EMAIL'] = input('GBIF email:')

In [5]:
# Query species
species_info = species.name_lookup("Bromus tectorum L.", rank='SPECIES')

# Get the first result
first_result = species_info['results'][0]

# Get the species key (speciesKey)
species_key = first_result['speciesKey']

# Check the result
first_result['species'], species_key

('Bromus tectorum', 264901674)

In [8]:
# Only download once
if not glob(str(project.project_dir / '*.csv')):
    # Only submit one request
    if not 'GBIF_DOWNLOAD_KEY' in os.environ:
        # Submit query to GBIF
        gbif_query = occ.download([
            "speciesKey = 102201512",
            "hasCoordinate = TRUE",
        ])
        os.environ['GBIF_DOWNLOAD_KEY'] = gbif_query[0]

    # Wait for the download to build
    download_key = os.environ['GBIF_DOWNLOAD_KEY']
    wait = occ.download_meta(download_key)['status']
    while not wait=='SUCCEEDED':
        wait = occ.download_meta(download_key)['status']
        time.sleep(5)

    # Download GBIF data
    download_info = occ.download_get(
        os.environ['GBIF_DOWNLOAD_KEY'], 
        path=project.project_dir)

    # Unzip GBIF data
    with zipfile.ZipFile(download_info['path']) as download_zip:
        download_zip.extractall(path=project.project_dir)
        
    # Clean up the .zip file
    shutil.rmtree(download_info[path])

# Find the extracted .csv file path (take the first result)
original_gbif_path = glob(str(project.project_dir / '*.csv'))[0]
original_gbif_path

INFO:Download file size: 516 bytes


FileNotFoundError: [Errno 2] No such file or directory: '/workspaces/data/cheatgrass-spread/0048089-260519110011954.zip'